In [1]:
import subprocess
import json
import sys
import tqdm
import pandas as pd

In [2]:
def run_test(command, data, model, mode) -> dict:
    proc = subprocess.run(
        [
            sys.executable, "measure_model.py",
            "--command", command,
            "--data", data,
            "--model", model,
            "--mode", mode
        ],
        capture_output=True,
        text=True
    )
    if proc.returncode != 0:
        raise RuntimeError(f"Subprocess failed: {proc.stderr}")
    print(proc.stdout)
    clean_lines = [line for line in proc.stdout.splitlines() if line.startswith("{") and line.endswith("}")]
    _dict = json.loads('\n'.join(clean_lines))  # <-- parse result
    assert isinstance(_dict, dict), "Expected a dictionary from the subprocess output"
    return _dict

def run_all_tests(data, model) -> dict:
    # macs
    results = []
    macs_n_params: dict = run_test("macs", data, model, "eval")
    for train in ["train", "eval"]:
        # flops, mem, time, energy
        flops = run_test("flops", data, model, train)
        mem = run_test("mem", data, model, train)
        time = run_test("time", data, model, train)
        energy = run_test("energy", data, model, train)
        results.append({
            "data": data,
            "model": model,
            "train": train,
            "macs": macs_n_params["macs"],
            "params": macs_n_params["params"],
            "flops": flops["flops"],
            "mem": mem["mem"],
            "time": time["time"],
            "energy": energy["energy"]
        })
    return results

In [3]:
run_all_tests("fetalAbdominal", "unet")

{"macs": 14975762432.0, "params": 68328066.0}

{"flops": 89835700224}

{"mem": 753236992}

{"time": 874.0086059570312}

[2026-01-12 19:39:28,012] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:39:28,012] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:39:28,013] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:39:28,013] [zeus.monitor.energy](energy.py:224) Monitoring CPU indices []
[2026-01-12 19:39:28,013] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
{"energy": 194.65000000223517}

{"flops": 7487881216}

{"mem": 190254592}

{"time": 174.9995574951172}

[2026-01-12 19:39:52,420] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:39:52,420] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:39:52,421] [zeus.monito

[{'data': 'fetalAbdominal',
  'model': 'unet',
  'train': 'train',
  'macs': 14975762432.0,
  'params': 68328066.0,
  'flops': 89835700224,
  'mem': 753236992,
  'time': 874.0086059570312,
  'energy': 194.65000000223517},
 {'data': 'fetalAbdominal',
  'model': 'unet',
  'train': 'eval',
  'macs': 14975762432.0,
  'params': 68328066.0,
  'flops': 7487881216,
  'mem': 190254592,
  'time': 174.9995574951172,
  'energy': 10.465999998152256}]

In [4]:
results = []
for model in tqdm.tqdm(["unet", "new_mednca", "transunet_b16", "segformer"]):
    results.extend(run_all_tests("fetalAbdominal", model))
for model in tqdm.tqdm(["vit4sgd", "densenet", "maxmednca_nobn"]):
    results.extend(run_all_tests("crc", model))

  0%|          | 0/4 [00:00<?, ?it/s]

{"macs": 14975762432.0, "params": 68328066.0}

{"flops": 89835700224}

{"mem": 753236992}

{"time": 874.0167846679688}

[2026-01-12 19:40:25,948] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:40:25,948] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:40:25,949] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:40:25,949] [zeus.monitor.energy](energy.py:224) Monitoring CPU indices []
[2026-01-12 19:40:25,949] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
{"energy": 173.43400000035763}

{"flops": 7487881216}

{"mem": 190254592}

{"time": 174.50294494628906}



 25%|██▌       | 1/4 [00:58<02:54, 58.19s/it]

[2026-01-12 19:40:50,621] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:40:50,621] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:40:50,622] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:40:50,622] [zeus.monitor.energy](energy.py:224) Monitoring CPU indices []
[2026-01-12 19:40:50,622] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
{"energy": 5.413000002503395}

{"macs": 444989440.0, "params": 7262.0}

{"flops": 15187443712}

{"mem": 415049728}

{"time": 1363.961669921875}

[2026-01-12 19:41:24,066] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:41:24,066] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:41:24,067] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:41:24,067] 

 50%|█████     | 2/4 [01:57<01:57, 58.58s/it]

...

[2026-01-12 19:44:54,343] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:44:54,343] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:44:54,344] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:44:54,344] [zeus.monitor.energy](energy.py:224) Monitoring CPU indices []
[2026-01-12 19:44:54,344] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
{"energy": 18.944000001996756}

{"macs": 531924480.0, "params": 6963081.0}

{"flops": 12315746304}

{"mem": 157665792}

{"time": 1465.9256591796875}

[2026-01-12 19:45:27,531] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:45:27,532] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:45:27,533] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:45:27,

 67%|██████▋   | 2/3 [01:59<00:59, 59.83s/it]

[2026-01-12 19:45:53,483] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:45:53,483] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:45:53,484] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:45:53,484] [zeus.monitor.energy](energy.py:224) Monitoring CPU indices []
[2026-01-12 19:45:53,484] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
{"energy": 25.79399999976158}

{"macs": 895058064.0, "params": 7027.0}

{"flops": 33322405248}

{"mem": 625625600}

{"time": 1518.159912109375}

[2026-01-12 19:46:26,254] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:46:26,255] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:46:26,256] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:46:26,256] 

100%|██████████| 3/3 [02:58<00:00, 59.55s/it]

[2026-01-12 19:46:52,229] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-01-12 19:46:52,229] [zeus.device.cpu.rapl](rapl.py:137) RAPL directory (/sys/class/powercap/intel-rapl) is available.
[2026-01-12 19:46:52,230] [zeus.monitor.energy](energy.py:223) Monitoring GPU indices [0].
[2026-01-12 19:46:52,230] [zeus.monitor.energy](energy.py:224) Monitoring CPU indices []
[2026-01-12 19:46:52,230] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
{"energy": 30.348999999463558}



In [5]:
results

[{'data': 'fetalAbdominal',
  'model': 'unet',
  'train': 'train',
  'macs': 14975762432.0,
  'params': 68328066.0,
  'flops': 89835700224,
  'mem': 753236992,
  'time': 874.0167846679688,
  'energy': 173.43400000035763},
 {'data': 'fetalAbdominal',
  'model': 'unet',
  'train': 'eval',
  'macs': 14975762432.0,
  'params': 68328066.0,
  'flops': 7487881216,
  'mem': 190254592,
  'time': 174.50294494628906,
  'energy': 5.413000002503395},
 {'data': 'fetalAbdominal',
  'model': 'new_mednca',
  'train': 'train',
  'macs': 444989440.0,
  'params': 7262.0,
  'flops': 15187443712,
  'mem': 415049728,
  'time': 1363.961669921875,
  'energy': 159.67599999904633},
 {'data': 'fetalAbdominal',
  'model': 'new_mednca',
  'train': 'eval',
  'macs': 444989440.0,
  'params': 7262.0,
  'flops': 827064320,
  'mem': 2672128,
  'time': 521.2802734375,
  'energy': 37.533999998122454},
 {'data': 'fetalAbdominal',
  'model': 'transunet_b16',
  'train': 'train',
  'macs': 2012636928.0,
  'params': 93230545.0

In [6]:
results = pd.DataFrame(results)

In [7]:
results.to_csv("measure_model_results.csv", index=False)

In [8]:
results["macs"] = results["macs"].apply(lambda x: f"{x/1e9:.2f} G")

In [9]:
results["flops"] = results["flops"].apply(lambda x: f"{x/1e9:.2f} G")


In [11]:
results

,data,model,train,macs,params,flops,mem,time,energy
0,fetalAbdominal,unet,train,14.98 G,68328066.0,89.84 G,753236992,874.016785,173.434
1,fetalAbdominal,unet,eval,14.98 G,68328066.0,7.49 G,190254592,174.502945,5.413
2,fetalAbdominal,new_mednca,train,0.44 G,7262.0,15.19 G,415049728,1363.961670,159.676
3,fetalAbdominal,new_mednca,eval,0.44 G,7262.0,0.83 G,2672128,521.280273,37.534
4,fetalAbdominal,transunet_b16,train,2.01 G,93230545.0,55.66 G,2123323392,2150.073486,439.463
5,fetalAbdominal,transunet_b16,eval,2.01 G,93230545.0,4.65 G,457813504,566.917480,58.979
6,fetalAbdominal,segformer,train,0.10 G,3711265.0,5.25 G,91485696,944.692200,65.082
7,fetalAbdominal,segformer,eval,0.10 G,3711265.0,0.21 G,27410432,274.174225,11.091
8,crc,vit4sgd,train,3.17 G,85628937.0,76.54 G,1731301888,1252.347900,359.386
9,crc,vit4sgd,eval,3.17 G,85628937.0,6.38 G,354008576,180.310852,18.944
